# Steps:
1. Get the vectore store

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [2]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


pdfEmbed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorStore = Chroma(persist_directory="./chroma_test1",collection_name="Example",embedding_function=pdfEmbed)


/var/folders/ql/ds02c6jn4kzbc08btxntjvhr0000gn/T/ipykernel_2291/367704537.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new down

2. Get the retriever

Similarity

In [3]:
retriever = vectorStore.as_retriever(search_type = "similarity",search_kwargs ={"k":3,"filter":{"page":{"$gte":10}}})#fetch_k=10 is not a parameter that standard similarity search necessarily uses in the way MMR does
answers = retriever.invoke(input="What is attention?")
# print(answers[0].page_content)

print([doc.metadata["page"] for doc in answers])

KeyboardInterrupt: 

"MMR promotes diversity among the retrieved documents while maintaining relevance. Therefore, it may return a different set from similarity search, but it can also return the same documents when those documents are both highly relevant and sufficiently diverse."

It controls the balance between:<br>

lambda_mult → 1<br>
     ↓<br>
more relevance<br>

<br>
lambda_mult → 0<br>
     ↓<br>
more diversity<br>

lambda_mult=0.8 -> Priority relevance
lambda_mult=0.2 -> prioritize diversity

In [ ]:
retriever = vectorStore.as_retriever(search_type = "mmr",search_kwargs ={"k":3,"fetch_k":10,"filter":{"page":{"$gte":10}},"lambda_mult":0.2 })#fetch_k=10 is not a parameter that standard similarity search necessarily uses in the way MMR does
answers = retriever.invoke(input="What is attention?")
# print(answers[0].page_content)

print([doc.metadata["page"] for doc in answers])

Similarity with threshold

In [ ]:
retriever = vectorStore.as_retriever(search_type = "similarity_score_threshold",search_kwargs ={"k":3,"filter":{"page":{"$gte":10}},"score_threshold":0.1})#fetch_k=10 is not a parameter that standard similarity search necessarily uses in the way MMR does
answers = retriever.invoke(input="What is attention?")
# print(answers[0].page_content)

print([doc.metadata["page"] for doc in answers])

for doc in answers:
    print(doc.id)
    print(doc.page_content)
    print(doc.metadata)

# Specialized Retrievers <br>
│<br>
├── MultiQueryRetriever<br>
├── ContextualCompressionRetriever<br>
├── ParentDocumentRetriever<br>
├── SelfQueryRetriever<br>
└── EnsembleRetriever<br>

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

MultiQueryRetriever uses an LLM to generate multiple reformulations of the user's query, retrieves documents for each reformulation using a base retriever, and combines the results.<br>

"How does attention work?"<br>
            ↓<br>
          LLM<br>
            ↓<br>
 ┌──────────┼──────────┐<br>
 ↓    &nbsp; &nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;    &nbsp;&nbsp;↓      &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;   &nbsp;&nbsp; ↓<br>
Query 1 &nbsp;&nbsp;  Query 2  &nbsp; &nbsp; Query 3<br>
            ↓<br>
       Base retriever<br>
            ↓<br>
          Chroma<br>
            ↓<br>
     Combine / deduplicate<br>
            ↓<br>
       answers<br>

In [ ]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace

llm = HuggingFaceEndpoint(repo_id="Qwen/Qwen2.5-7B-Instruct",temperature=0)
chat_model = ChatHuggingFace(llm=llm)

multi_query_retriever = MultiQueryRetriever.from_llm(retriever=retriever,llm= chat_model)



run_manager is a LangChain callback manager passed through retriever methods so that nested operations such as LLM calls can participate in the same tracing, logging, and callback lifecycle.<br>
questions = multi_query_retriever.generate_queries(question='What is attention?',run_manager=)

invoke for multiquery

In [ ]:
answers = multi_query_retriever.invoke(input="How is attention?")
print(len(answers))
print([doc.page_content for doc in answers])

Similarity: one query → retrieve relevant chunks.<br>

MultiQuery: one question → multiple query formulations → retrieve from each → combine unique results.<br>

MultiQueryRetriever improves recall by reformulating the user's query into multiple perspectives.

In [ ]:
base_answers = retriever.invoke("How does attention work?")

multi_answers = multi_query_retriever.invoke("How does attention work?")

print("Base:", len(base_answers))
print("MultiQuery:", len(multi_answers))

print("Base pages:",
      [d.metadata.get("page") for d in base_answers])

print("MultiQuery pages:",
      [d.metadata.get("page") for d in multi_answers])

# Coontext compression retriever

In [ ]:
#"filter":{"page":{"$gte":10}}
baseRetriever = vectorStore.as_retriever(search_type = "similarity",search_kwargs ={"k":5})#fetch_k=10 is not a parameter that standard similarity search necessarily uses in the way MMR does
answers = retriever.invoke(input="What is attention?")
# print(answers[0].page_content)

print([doc.metadata["page"] for doc in answers])

MultiQuery changes the query before retrieval, while Contextual Compression processes the retrieved documents after retrieval.<br>
Metadata filter happens BEFORE retrieval.<br>

Contextual compression happens AFTER retrieval

LLMChainExtractor uses your LLM to read each retrieved document together with the user's question and extract only the parts of that document that are relevant to the question.<br>
Document<br>
   ↓<br>
LLM reads text<br>
   ↓<br>
Extract relevant text<br>
   ↓<br>
New Document with smaller page_content<br>

ContextualCompressionRetriever first retrieves candidate documents using a base retriever, then uses a compressor such as LLMChainExtractor to extract only the query-relevant portions of those documents before passing them to the LLM.<br>
Contextual compression reduces the retrieved context by extracting only query-relevant content; documents with no relevant content may also be removed.

In [ ]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm=chat_model)
contextCompRetriever = ContextualCompressionRetriever(base_retriever=baseRetriever,base_compressor=compressor)
contextAnswers = contextCompRetriever.invoke("How does attention work?")
print("contextAnswers:", len(contextAnswers))
print("contextAnswers:",
      [d.metadata.get("page") for d in contextAnswers])


# parent document retriever
ParentDocumentRetriever retrieves small child chunks for precise semantic search, then uses their parent IDs to return larger parent documents to the LLM, providing better context while maintaining accurate retrieval.<br>

pdfDocs <br>
   ↓<br>
parent_splitter<br>
   ↓<br>
Parent chunks<br>
   ↓<br>
child_splitter<br>
   ↓<br>
Child chunks<br>
   ↓<br>
Embed child chunks<br>
   ↓<br>
Chroma<br>

# Note :It can store documents and metadata, but ParentDocumentRetriever is designed with a separate document store for the parent documents.
          ParentDocumentRetriever 
                         │
             ┌───────────┴───────────┐
             ↓                       ↓
       vectorstore                docstore
       (Chroma)                 (InMemoryStore)
             │                       │
       CHILD chunks              PARENT chunks
       + embeddings              + original text

that's an important distinction. InMemoryDocstore is the one commonly used with FAISS, while InMemoryStore is used with ParentDocumentRetriever<br>
Child has its own ID; parent has its own ID. The child's metadata contains the parent's ID so the retriever can connect them.

Component	========>    Persistence <br>
Chroma	==========>  Persistent for the collection<br>
InMemoryStore ====>	Memory only; recreated when code runs again<br>

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.stores import InMemoryStore

#1. load the doc
loader = PyPDFLoader(file_path="attention.pdf")
pdfDocs = loader.load()
print(len(pdfDocs))

#2. parent splitter and child splitter
seperators = ["\n\n","\n"," ",""]
parentSplitter = RecursiveCharacterTextSplitter(separators=seperators,chunk_size=1000,chunk_overlap=100)
childSplitter = RecursiveCharacterTextSplitter(separators=seperators,chunk_size=200,chunk_overlap=50)

#3. Embedding
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#4. Vector store
vectorStore = Chroma(collection_name="parent_child2",embedding_function=embedding)

#5. In memery docstore for storing parent chunks, chromonly stores child chunks
docStore = InMemoryStore()

#6. retriever
retriever = ParentDocumentRetriever(child_splitter=childSplitter,parent_splitter=parentSplitter,docstore=docStore,vectorstore=vectorStore,search_kwargs={"k": 3})

retriever.add_documents(pdfDocs)

docs = retriever.invoke("What is Attention?")

print(vectorStore._collection.count())
print(len(docStore.store))
print(len(docs))


In [ ]:
child_docs = vectorStore.similarity_search(
    "What is Attention?",
    k=3
)

print(type(child_docs[0]))
for doc in child_docs:
    print("CONTENT:")
    print(doc.page_content[:100])
    print("METADATA:")
    print(doc.metadata)
    print()

In [ ]:
print("Child IDs:")
for doc in child_docs:
    print(doc.metadata["doc_id"])

print("\nParent IDs:")
for key in docStore.store.keys():
    print(key)

# SelfQueryRetriever

document retrieval by combining semantic search with structured filtering.

It converts a natural-language query into a semantic query + metadata filters.<br>

User asks:<br>

"Find transformer documents from 2024 after page 10."<br>

SelfQueryRetriever can determine:<br>

Semantic query:<br>
"transformer"<br>


Metadata filter:<br>
year = 2024<br>
page > 10<br>

Then it sends that structured query to the vector store.<br>

Normal similarity:<br>
Question → embedding → vector search<br>
<br>
Self-query:<br>
Question<br>
   ↓<br>
LLM<br>
   ↓<br>
semantic query + metadata filter<br>
   ↓<br>
vector store<br>

Note : A structured_query_translator parameter allows for adapting the retriever to different vector stores by translating internal query formats into database-specific search parameters.


In [ ]:
import langchain_chroma

print(langchain_chroma.__file__)

In [ ]:
from langchain_chroma import Chroma

print(Chroma)

In [ ]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever

ImportError: cannot import name 'DatabricksVectorSearch' from 'langchain_community.vectorstores' (/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain/venv/lib/python3.11/site-packages/langchain_community/vectorstores/__init__.py)

That traceback is a version-mismatch issue, not something wrong in your code. langchain_classic's self_query/base.py expects langchain_community.vectorstores to export DatabricksVectorSearch, but your installed langchain_community is a version that doesn't have it (either too old, or renamed/moved in a newer release).

Ran 5 commands

Confirmed — this is a genuine bug in langchain-classic 1.0.8, not a version mismatch you caused. Its _get_builtin_translator() function tries to import DatabricksVectorSearch from langchain_community.vectorstores, but that class isn't exported there even in the newest langchain-community (0.4.2) — so no combination of current versions will work through that code path. Since you're using Chroma anyway, you don't need most of those translators.

Workaround — bypass from_llm's auto-translator lookup by passing the Chroma translator directly:from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

In [ ]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_chroma import Chroma as langChroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

llm = HuggingFaceEndpoint(repo_id="Qwen/Qwen2.5-7B-Instruct",temperature=0)
chat_model = ChatHuggingFace(llm=llm)

metdata_attribute_info = [
        AttributeInfo(
        name="genre",
        description="The genre of the movie",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="A 1-10 rating for the movie",
        type="float",
    ),
]

document_content_description = "Brief summary of a movie"

# Add documents with metadata to the vector store
documents = [
    Document(
        page_content="A space adventure with aliens",
        metadata={"genre": "science fiction", "rating": 8.5},
    ),
    Document(
        page_content="A comedy about life in a small town",
        metadata={"genre": "comedy", "rating": 7.0},
    ),
]

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# vectorStoreChroma.delete_collection()

vectorStoreChroma = langChroma(embedding_function=embedding)
# vectorStoreChroma.add_documents(documents=documents)
vectorStoreChroma.add_documents(documents = documents,ids=["doc_1", "doc_2"])

retriever = SelfQueryRetriever.from_llm(llm =chat_model,vectorstore=vectorStoreChroma,document_contents=document_content_description,metadata_field_info=metdata_attribute_info,
structured_query_translator=ChromaTranslator())
result = retriever.invoke("science fiction movie with rating greater than 8")
print(result)
print(len(result))

# Ensembele retrievers

The EnsembleRetriever in LangChain takes a list of different retrievers, runs them together, and merges their results using Reciprocal Rank Fusion (RRF). It lets you combine a keyword search (like BM25) with a semantic vector search to improve RAG accuracy
<br>
How It WorksCombines methods: <br>
1. Merges sparse (keyword) and dense (vector) search results.<br>
2. Reranks output: Uses RRF(Reciprocal Rank Fusion (RRF)) to score and sort documents based on their ranks across all input retrievers.<br>
3. Applies weights: Lets you assign percentage weights to each retriever to control their influence<br>
4. Runtime Configuration: The retriever supports dynamic configuration, allowing users to adjust parameters (e.g., the number of documents to retrieve) at runtime.<br>



This approach, often referred to as "hybrid search," integrates sparse retrievers (e.g., BM25) and dense retrievers (e.g., Chroma or FAISS) to provide a more comprehensive and accurate retrieval system.



In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Sample documents
doc_list_1 = [
    "I like apples",
    "I like oranges",
    "Apples and oranges are fruits",
]

doc_list_2 = [
    "You like apples",
    "You like oranges",
]

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
bm25Retriever = BM25Retriever.from_texts(texts=doc_list_1,metadatas=[{"souce":1}]*len(doc_list_1))
bm25Retriever.k = 2
vectorStoreEnemble = Chroma.from_texts(texts=doc_list_2,metadatas=[{"source":2}]*len(doc_list_2),embedding=embedding)
baseRetrieverEnsemble = vectorStoreEnemble.as_retriever(search_type="similarity",search_kwargs={"k":2})

ensemble_retriever = EnsembleRetriever(retrievers=[bm25Retriever,baseRetrieverEnsemble],weights=[0.5,0.5])
# Retrieve documents
docs = ensemble_retriever.invoke("apples")
print(docs)
print([doc.metadata for doc in docs])
print(len(docs))

In [ ]:
print("BM25 alone:", bm25Retriever.invoke("apples"))
print("Vector alone:", baseRetrieverEnsemble.invoke("apples"))
print("Chroma collection count:", vectorStoreEnemble._collection.count())